In [5]:
!pip install pandas

In [12]:
import pandas as pd

In [14]:
df = pd.read_csv("spotify_millsongdata.csv")

In [15]:
df.head(5)

,artist,song,link,text
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \r\nA..."
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \r\nTouch me gen..."
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \r\nWhy I had...
3,ABBA,Bang,/a/abba/bang_20598415.html,Making somebody happy is a question of give an...
4,ABBA,Bang-A-Boomerang,/a/abba/bang+a+boomerang_20002668.html,Making somebody happy is a question of give an...


In [16]:
df.tail(5)

,artist,song,link,text
57645,Ziggy Marley,Good Old Days,/z/ziggy+marley/good+old+days_10198588.html,Irie days come on play \r\nLet the angels fly...
57646,Ziggy Marley,Hand To Mouth,/z/ziggy+marley/hand+to+mouth_20531167.html,Power to the workers \r\nMore power \r\nPowe...
57647,Zwan,Come With Me,/z/zwan/come+with+me_20148981.html,all you need \r\nis something i'll believe \...
57648,Zwan,Desire,/z/zwan/desire_20148986.html,northern star \r\nam i frightened \r\nwhere ...
57649,Zwan,Heartsong,/z/zwan/heartsong_20148991.html,come in \r\nmake yourself at home \r\ni'm a ...


In [17]:
df.shape

(57650, 4)

In [18]:
df.isnull().sum()

artist    0
song      0
link      0
text      0
dtype: int64

In [20]:
df= df.sample(5000).drop('link', axis=1).reset_index(drop=True)

In [26]:
df.head(5)

,artist,song,text
0,Grateful Dead,Operator,"Operator, can you help me \r\nHelp me if you ..."
1,Michael W. Smith,The Wonderful Cross,When I survey the wondrous cross \r\nOn which...
2,Bette Midler,I Remember You,I remember you. \r\nYou're the one who made m...
3,Peter Cetera,I Wasn't The One,Your eyes and my eyes haven't talked in quite ...
4,"Harry Connick, Jr.",Maybe,"Maybe far away, or maybe real near by. \r\nHe..."


In [28]:
df['text'][0]

"Operator, can you help me  \r\nHelp me if you please  \r\nGive me the right area code  \r\nAnd the number that I need  \r\nMy rider left upon the Midnight Flyer  \r\nSingin' like a summer breeze  \r\n  \r\nI think she's somewhere down south  \r\nDown about Baton Rouge  \r\nBut I just a can't remember no number  \r\nA number I can use  \r\nDirectory don't have it  \r\nCentral done forgot it  \r\nGotta find a number to use.  \r\n  \r\nTrying to check out her number  \r\nTrying to run down her line.  \r\nOperator said that's priv'ledged information  \r\nAnd it ain't no business of mine  \r\nIt's floodin' down in Texas  \r\nPoles are out in Utah  \r\nGotta find a private line  \r\n  \r\nShe could be hangin' 'round the steel mill  \r\nWorking in a house of blue lights  \r\nRiding a getaway bus out of Portland  \r\nTalking to the night  \r\nI don't know where she's going  \r\nI don't care where she's been  \r\nLong as she's been doin' it right  \r\n  \r\nLong as she's been doin' it right\r\

In [30]:
df.shape

(5000, 3)

Text Cleaning/ Text Processing 

In [33]:
df['text']= df['text'].str.lower().replace(r'^\w\s',' ').replace(r'\n', ' ', regex=True)

In [35]:
 !pip install nltk

In [36]:
import nltk
nltk.download('punkt')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\kassab\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [68]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\kassab\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [70]:
from nltk.stem.porter import PorterStemmer
from nltk.tokenize import word_tokenize
import nltk

stemmer = PorterStemmer()

def token(txt):
    tokens = word_tokenize(txt)
    a = [stemmer.stem(w) for w in tokens]
    return " ".join(a)

print(token("you are beautiful, beauti" ))

you are beauti , beauti


In [72]:
df['text'].apply(lambda x: token(x))

0       oper , can you help me help me if you pleas gi...
1       when i survey the wondrou cross on which the p...
2       i rememb you . you 're the one who made my dre...
3       your eye and my eye have n't talk in quit a wh...
4       mayb far away , or mayb real near by . he may ...
                              ...                        
4995    babi , look here at me have you ever seen me t...
4996    alright now ! ! wo n't you listen ? when i fir...
4997    if you want the sun . then i 'll be there to m...
4998    stand here alon with you wonder what it is tha...
4999    i 'm a welsh nobl , i mobilis wale against the...
Name: text, Length: 5000, dtype: object

In [73]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [74]:
tfid = TfidfVectorizer(analyzer='word', stop_words='english')

In [75]:
matrix= tfid.fit_transform(df['text'])

In [76]:
similer = cosine_similarity(matrix)

In [77]:
similer[0]

array([1.        , 0.        , 0.06375348, ..., 0.01606529, 0.0112896 ,
       0.00940489])

In [78]:
df[df['song']=='In Your Eyes'].index[0]

2210

Recommender Function 

In [80]:
def recommender(song_name):
    idx = df[df['song']== song_name].index[0]
    distance = sorted(list(enumerate(similer[idx])), reverse=True , key = lambda x:x[1])
    song = []
    for s_id in distance[1:21]:
        song.append(df.iloc[s_id[0]].song)
    return song   

In [81]:
recommender("In Your Eyes")

['Dream',
 'Dream Baby Dream',
 'Dream',
 '(all I Can Do Is) Dream You',
 "Walkin' Dream",
 'Goodbye',
 "This Time The Dream's On Me",
 'Better Than A Dream (feat. Judy Holliday)',
 'Did You Ever Have A Dream',
 'Enter My Dream',
 'Dream I Killed God',
 'Perchance To Dream',
 'Dream A Little Dream Of Me',
 'Dream On Little Dreamer',
 'Every Hand In The Land',
 'Eyes Of Innocence',
 'In The End (Radio Version)',
 'Ballad Of A Teenage Queen',
 "Surf's Up... School's Out",
 'Last Night']

In [82]:
import pickle

In [83]:
pickle.dump(similer,open("similarity", "wb"))

In [84]:
pickle.dump(df,open("similarity", "wb"))

In [85]:
with open("df.pkl", "wb") as f:
    pickle.dump(df, f)

In [87]:
pickle.dump(similer, open('similarity.pkl', 'wb'))